# Generate COMPASS platinum-arm figures

One notebook for both supported COMPASS PROFILE survival-analysis arms. Run the shared setup once, then run either the first-treatment or treatment-anchored section independently.


## Shared setup


## Imports


In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Patch

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})


## Clinical category mapping

40 labs in six buckets. `Body height` is dropped due to data-quality concerns. PT is folded into LFT since it tracks hepatic synthetic function. The same palette is reused across all three figures.


In [ ]:
CBC = {
    "WBC", "RBC", "Hemoglobin", "Hematocrit",
    "MCV", "MCH", "MCHC", "RDW", "Platelets",
    "Neutrophils absolute", "Lymphocytes absolute",
    "Monocytes absolute", "Eosinophils absolute", "Basophils absolute",
}
CMP = {"Sodium", "Potassium", "Chloride", "CO2",
       "BUN", "Creatinine", "Glucose", "Calcium"}
LFT = {"ALT", "AST", "Alkaline phosphatase",
       "Total bilirubin", "Direct bilirubin",
       "Albumin", "Globulin", "Total protein", "PT"}
VITALS = {"Body weight", "Body temperature", "Heart rate",
          "Respiratory rate", "Systolic blood pressure",
          "Diastolic blood pressure"}
ANDROGEN = {"PSA", "Testosterone"}
OTHER = {"TSH"}
DROP = {"Body height"}

CATEGORY_MAP = {}
for label, members in [
    ("CBC", CBC), ("CMP", CMP), ("LFT", LFT),
    ("Vitals", VITALS), ("Androgen axis", ANDROGEN), ("Other", OTHER),
]:
    for m in members:
        CATEGORY_MAP[m] = label

DRAW_ORDER = ["Other", "Vitals", "CMP", "LFT", "CBC", "Androgen axis"]
LEGEND_ORDER = ["Androgen axis", "CBC", "LFT", "CMP", "Vitals", "Other"]

CATEGORY_COLORS = {
    "Androgen axis": "#8e1c2b",
    "CBC":           "#16a085",
    "LFT":           "#e67e22",
    "CMP":           "#7d3c98",
    "Vitals":        "#5d6d7e",
    "Other":         "#95a5a6",
}
NS_COLOR = "#d5d8dc"


def assign_category(lab_name: str) -> str:
    return CATEGORY_MAP.get(lab_name, "Other")


def format_label(lab_name, feature_stat):
    if not feature_stat or pd.isna(feature_stat) or feature_stat == "":
        return lab_name
    return f"{lab_name} ({feature_stat})"


def parse_feature(name):
    """Split 'LAB_NAME__stat' into (lab_name, stat). Returns (name, '') if no '__'."""
    if "__" in name:
        lab, stat = name.split("__", 1)
        return lab, stat
    return name, ""


## First-treatment arm

Uses the default `t_first_treatment` outputs under `prediction_inputs/` and `local_runs/`.


## Paths

`BASE` points at the directory containing `cox/landmark_{lm}/...`, `xgboost/landmark_{lm}/...`, and `deephit/landmark_{lm}/...` outputs. `LONGITUDINAL_CSV` and `INPUTS_DIR` feed Figure 1 (cohort overview). Override all for local copies.


In [ ]:
NEPC_PROJ_PATH = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS")

BASE = NEPC_PROJ_PATH / "survival_analysis" / "local_runs"
LONGITUDINAL_CSV = NEPC_PROJ_PATH / "longitudinal_prediction_data.csv"
INPUTS_DIR = NEPC_PROJ_PATH / "survival_analysis" / "prediction_inputs"
AGGREGATED_CSV = INPUTS_DIR / "aggregated_landmark0.csv"  # optional stage enrichment for Table 1

OUT_DIR = Path("/data/gusev/USERS/jpconnor/figures/CAIA/COMPASS")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LANDMARKS = [0, 90]
TOP_N = 15  # number of features shown per importance panel


## Figure 1 — Cohort overview

Three panels — each saved standalone, then assembled into one compiled image — plus a companion Table 1 (baseline characteristics, written separately as CSV + markdown, not a plotted panel):

- **Panel A — CONSORT attrition.** Reads the structured attrition counts `longitudinal_data_processing.py` writes to `cohort_attrition.json` (next to `LONGITUDINAL_CSV`) and `build_prediction_inputs.py` writes to `landmark_attrition.json` (in `INPUTS_DIR`) — no log-scraping.
- **Panel B — KM curve.** Platinum-free survival for the whole cohort (not landmark-rebased), from `LONGITUDINAL_CSV`.
- **Panel C — Timing histograms.** Record span, diagnosis → first treatment, and time to platinum (event patients only).

Table 1 covariates are limited to what PROFILE actually carries for the platinum analysis: Age, platinum exposure, follow-up time, plus optional cancer stage if `AGGREGATED_CSV` has `CANCER_STAGE_*` columns (only present if the pipeline run passed `--stage-file`). There is no race/ethnicity/insurance field in PROFILE.


In [ ]:
COHORT_LABEL = "PROFILE"
STAGE_COLUMNS = ["CANCER_STAGE_II", "CANCER_STAGE_III", "CANCER_STAGE_IV"]


def load_profile_patient_and_labs(path, *, id_col="DFCI_MRN"):
    """(patient_df, labs_df) split of longitudinal_prediction_data.csv."""
    df = pd.read_csv(path, low_memory=False)
    for c in ["DIAGNOSIS_DATE", "FIRST_TREATMENT_DATE", "PLATINUM_DATE",
              "LAST_CONTACT_DATE", "LAB_DATE", "FIRST_RECORD_DATE"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    if "t_dx_to_tx" not in df.columns and {"DIAGNOSIS_DATE", "FIRST_TREATMENT_DATE"} <= set(df.columns):
        df["t_dx_to_tx"] = (df["FIRST_TREATMENT_DATE"] - df["DIAGNOSIS_DATE"]).dt.days.astype("Float64")

    patient_level = [
        id_col, "AGE_AT_TREATMENTSTART", "FIRST_RECORD_DATE", "DIAGNOSIS_DATE",
        "FIRST_TREATMENT_DATE", "FIRST_TREATMENT", "LAST_CONTACT_DATE",
        "DEATH", "PLATINUM_MEDICATION", "PLATINUM_DATE", "PLATINUM",
        "t_diagnosis", "t_first_treatment", "t_treatment_anchor", "t_platinum",
        "t_last_contact", "t_death", "t_dx_to_tx",
    ]
    pat_cols = [c for c in patient_level if c in df.columns]
    patient_df = df[pat_cols].drop_duplicates(subset=[id_col]).reset_index(drop=True)

    lab_cols = [c for c in [id_col, "LAB_NAME", "LAB_VALUE", "LAB_UNIT", "LAB_DATE", "t_lab"]
                if c in df.columns]
    labs_df = df.loc[df["LAB_NAME"].notna(), lab_cols].reset_index(drop=True)
    return patient_df, labs_df


def record_span_days(labs_df, *, id_col, date_col="LAB_DATE"):
    g = labs_df.groupby(id_col)[date_col]
    span = (g.max() - g.min()).dt.days.astype("Float64")
    span.name = "record_span_days"
    return span


def load_attrition(longitudinal_csv, inputs_dir):
    cohort_path = longitudinal_csv.parent / "cohort_attrition.json"
    landmark_path = inputs_dir / "landmark_attrition.json"
    if not cohort_path.exists():
        raise FileNotFoundError(
            f"{cohort_path} not found -- re-run longitudinal_data_processing.py "
            "(it now writes cohort_attrition.json alongside its CSV outputs)."
        )
    if not landmark_path.exists():
        raise FileNotFoundError(
            f"{landmark_path} not found -- re-run build_prediction_inputs.py "
            "(it now writes landmark_attrition.json alongside its CSV outputs)."
        )
    return {**json.loads(cohort_path.read_text()), **json.loads(landmark_path.read_text())}


def render_consort_panel(ax, attrition):
    steps = [
        ("Patients with usable labs/vitals", attrition["n_with_labs"]),
        ("+ in death/last-contact table", attrition["n_after_death_table_join"]),
        ("+ recorded first treatment", attrition["n_after_first_treatment_filter"]),
        (f"+ ≥{attrition['min_psa_count']} PSA records", attrition["n_after_psa_count_filter"]),
        ("− PARPi-exposed excluded", attrition["n_after_parpi_exclusion"]),
    ]
    for landmark_str, n in sorted(attrition["eligible_by_landmark"].items(), key=lambda kv: int(kv[0])):
        sign = "+" if int(landmark_str) > 0 else ""
        steps.append((f"Eligible at landmark {sign}{landmark_str}d", n))
    steps.append(("Common cohort (all landmarks)", attrition["n_common_across_landmarks"]))

    n_steps = len(steps)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, n_steps + 0.4)
    ax.invert_yaxis()
    ax.axis("off")
    ax.set_title("Cohort attrition (CONSORT)", fontsize=12.5, weight="bold")

    box_w, box_h = 0.86, 0.62
    for i, (label, n) in enumerate(steps):
        y = i + (1 - box_h) / 2
        ax.add_patch(FancyBboxPatch(
            ((1 - box_w) / 2, y), box_w, box_h,
            boxstyle="round,pad=0.02,rounding_size=0.04",
            facecolor="#eef1f5", edgecolor="#5d6d7e", linewidth=1.0,
        ))
        ax.text(0.5, y + box_h / 2, f"{label}\nn = {n:,}",
                ha="center", va="center", fontsize=9)
        if i < n_steps - 1:
            ax.annotate(
                "", xy=(0.5, i + 1 + (1 - box_h) / 2), xytext=(0.5, y + box_h),
                arrowprops=dict(arrowstyle="-|>", color="#5d6d7e", lw=1.2),
            )

    split_sizes = attrition["split_sizes"]
    footer = "  ·  ".join(f"{k}: n={v:,}" for k, v in split_sizes.items())
    ax.text(0.5, n_steps + 0.15, f"Train/valid/test split — {footer}",
            ha="center", va="top", fontsize=8.5, color="#5d6d7e", style="italic")


def render_km_panel(ax, patient_df):
    from lifelines import KaplanMeierFitter

    dur = pd.to_numeric(patient_df["t_platinum"], errors="coerce")
    evt = pd.to_numeric(patient_df["PLATINUM"], errors="coerce")
    mask = dur.notna() & evt.notna() & (dur >= 0)
    kmf = KaplanMeierFitter(label=f"{COHORT_LABEL} (n={int(mask.sum()):,})")
    kmf.fit(durations=dur.loc[mask].to_numpy(dtype=float),
            event_observed=evt.loc[mask].to_numpy(dtype=float))
    kmf.plot_survival_function(ax=ax, color="#1f3a93", ci_show=True)
    ax.set_title("Platinum-free survival")
    ax.set_xlabel("Days from first record")
    ax.set_ylabel("Platinum-free probability")
    ax.set_ylim(0, 1.02)


def _overlay_hist(ax, series, *, bins=50):
    v = pd.to_numeric(series, errors="coerce").dropna().to_numpy(dtype=float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        ax.text(0.5, 0.5, "(no data)", ha="center", va="center", transform=ax.transAxes)
        return
    if isinstance(bins, int):
        lo, hi = np.percentile(v, [0.5, 99.5])
        if hi <= lo:
            lo, hi = v.min(), v.max() + 1.0
        bins = np.linspace(lo, hi, bins + 1)
    ax.hist(v, bins=bins, density=True, alpha=0.75, color="#1f3a93",
            label=f"{COHORT_LABEL} (n={v.size:,})", edgecolor="white", linewidth=0.3)
    ax.legend(fontsize=9, loc="best")


def render_timing_panel(axes, patient_df, labs_df, id_col):
    ax_span, ax_dx2tx, ax_tplat = axes

    span = record_span_days(labs_df, id_col=id_col)
    _overlay_hist(ax_span, span, bins=50)
    ax_span.set_xlabel("Record span (days)")
    ax_span.set_ylabel("Density")
    ax_span.set_title("Per-patient lab record span")

    _overlay_hist(ax_dx2tx, patient_df.get("t_dx_to_tx", pd.Series(dtype=float)), bins=50)
    ax_dx2tx.set_xlabel("Days: diagnosis → first treatment")
    ax_dx2tx.set_ylabel("Density")
    ax_dx2tx.set_title("Diagnosis → first treatment")

    is_event = pd.to_numeric(patient_df["PLATINUM"], errors="coerce").fillna(0).astype(int).astype(bool)
    _overlay_hist(ax_tplat, patient_df.loc[is_event, "t_platinum"], bins=40)
    ax_tplat.set_xlabel("Days to platinum, from first record (events only)")
    ax_tplat.set_ylabel("Density")
    ax_tplat.set_title("Time to platinum (event patients only)")


In [ ]:
def _mean_sd(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    return f"{s.mean():.1f} ± {s.std():.1f}" if not s.empty else "n/a"


def _median_iqr(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return "n/a"
    q1, med, q3 = s.quantile([0.25, 0.5, 0.75])
    return f"{med:.1f} ({q1:.1f}–{q3:.1f})"


def _count_pct(mask, total):
    n = int(mask.sum())
    return f"{n:,} ({100 * n / total:.1f}%)" if total else "n/a"


def _reconstruct_stage(aggregated_df):
    if not set(STAGE_COLUMNS).issubset(aggregated_df.columns):
        return None
    onehot = aggregated_df[STAGE_COLUMNS].apply(pd.to_numeric, errors="coerce")
    unknown = onehot.isna().all(axis=1)
    any_hit = (onehot.fillna(0) > 0).any(axis=1)
    picked = onehot.fillna(0).idxmax(axis=1).str[len("CANCER_STAGE_"):]
    stage = picked.where(any_hit, "I")
    return stage.mask(unknown, "Unknown")


def build_table1(patient_df, aggregated_df):
    n = len(patient_df)
    rows = [("N", f"{n:,}")]

    age = patient_df.get("AGE_AT_TREATMENTSTART", pd.Series(dtype=float))
    rows.append(("Age at first treatment, mean ± SD", _mean_sd(age)))
    rows.append(("Age at first treatment, median (IQR)", _median_iqr(age)))

    rows.append(("Platinum exposure, n (%)",
                 _count_pct(pd.to_numeric(patient_df["PLATINUM"], errors="coerce").fillna(0).astype(bool), n)))
    rows.append(("Median follow-up from first record, days (IQR)",
                 _median_iqr(patient_df.get("t_last_contact", pd.Series(dtype=float)))))

    if aggregated_df is None:
        rows.append(("Cancer stage", "not available (no AGGREGATED_CSV with CANCER_STAGE_* columns)"))
    else:
        n_agg = len(aggregated_df)
        stage = _reconstruct_stage(aggregated_df)
        if stage is not None:
            for cat in ["I", "II", "III", "IV", "Unknown"]:
                rows.append((f"  Cancer stage {cat}, n (%)", _count_pct(stage.eq(cat), n_agg)))
        else:
            rows.append(("Cancer stage", "not available in AGGREGATED_CSV"))

    return pd.DataFrame(rows, columns=["Characteristic", "Value"])


def _to_markdown_table(df):
    """Minimal GFM table renderer (no `tabulate` dependency, which isn't in
    this repo's pinned dependency list -- see README)."""
    header = "| " + " | ".join(df.columns) + " |"
    sep = "| " + " | ".join("---" for _ in df.columns) + " |"
    body = "\n".join("| " + " | ".join(str(v) for v in row) + " |"
                     for row in df.itertuples(index=False))
    return "\n".join([header, sep, body])


def write_table1(table1, out_base):
    out_base.parent.mkdir(parents=True, exist_ok=True)
    table1.to_csv(out_base.with_suffix(".csv"), index=False)
    out_base.with_suffix(".md").write_text(_to_markdown_table(table1))
    return [out_base.with_suffix(".csv"), out_base.with_suffix(".md")]


In [ ]:
ID_COL = "DFCI_MRN"

print(f"Loading attrition counts from {LONGITUDINAL_CSV.parent} and {INPUTS_DIR} ...")
attrition = load_attrition(LONGITUDINAL_CSV, INPUTS_DIR)

print(f"Loading {LONGITUDINAL_CSV} ...")
patient_df, labs_df = load_profile_patient_and_labs(LONGITUDINAL_CSV, id_col=ID_COL)
print(f"  patients={len(patient_df):,}  labs={len(labs_df):,}")

aggregated_df = None
if AGGREGATED_CSV.exists():
    aggregated_df = pd.read_csv(AGGREGATED_CSV, low_memory=False)
    print(f"Loaded {AGGREGATED_CSV} for Table 1 enrichment: {len(aggregated_df):,} rows")
else:
    print(f"[note] {AGGREGATED_CSV} not found; Table 1 will omit cancer stage")

# --- Panel A: CONSORT ---
n_landmarks = len(attrition["eligible_by_landmark"])
fig, ax = plt.subplots(figsize=(7.5, 0.62 * (n_landmarks + 8) + 1))
render_consort_panel(ax, attrition)
fig.tight_layout()
for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure1a_consort.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

# --- Panel B: KM curves ---
fig, ax = plt.subplots(figsize=(6.5, 4.8))
render_km_panel(ax, patient_df)
fig.tight_layout()
for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure1b_km.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

# --- Panel C: timing histograms ---
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
render_timing_panel(axes, patient_df, labs_df, ID_COL)
fig.tight_layout()
for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure1c_timing.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

# --- Compiled Figure 1 ---
fig = plt.figure(figsize=(14, 6 + 0.62 * (n_landmarks + 8)))
gs = fig.add_gridspec(3, 3, height_ratios=[0.62 * (n_landmarks + 8), 1.0, 1.0])
render_consort_panel(fig.add_subplot(gs[0, :]), attrition)
render_km_panel(fig.add_subplot(gs[1, :]), patient_df)
ax_span, ax_dx2tx, ax_tplat = (fig.add_subplot(gs[2, 0]), fig.add_subplot(gs[2, 1]), fig.add_subplot(gs[2, 2]))
render_timing_panel((ax_span, ax_dx2tx, ax_tplat), patient_df, labs_df, ID_COL)
fig.suptitle("Figure 1 — Cohort overview", fontsize=14, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.98])
for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure1_cohort_overview.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

# --- Table 1 ---
table1 = build_table1(patient_df, aggregated_df)
for p in write_table1(table1, OUT_DIR / "table1_baseline_characteristics"):
    print(f"wrote {p}")
table1


## Figure 3 — Paired volcano (univariate Cox)

Each dot is one (lab × feature_stat) pair. ns features (q ≥ 0.05) are faded gray; q-significant features are colored by their clinical category. Three narrative beats highlighted on the figure:

- **Testosterone** is a key indicator at both landmarks
- **PSA** becomes significant at +90d
- **CBC + LFT** signals (general health markers) emerge by +90d

Source: `cox/landmark_{lm}/both/cox_agg_univariate_nobs_adjusted.csv`.


In [ ]:
# ----------------------------- labeling knobs ---------------------------
TOP_K_PER_PANEL = 4
ALWAYS_LABEL = {"Hemoglobin", "Albumin", "Alkaline phosphatase"}
LABEL_FORMAT = "{lab} ({stat})"
LABEL_FONTSIZE = 8.5
MIN_LABEL_SPACING_NLP = 0.55
PANEL_XLIM = (-1.5, 1.5)
Y_MAX_CAP = 30  # -log10(p) ceiling; anything above this is drawn at the cap as a triangle


def q_threshold_neglog10p(sub):
    """y-value (-log10 p) at which q just crosses 0.05; None if no q-sig features."""
    sig = sub.loc[sub["q_value"] < 0.05, "p_value"]
    if sig.empty:
        return None
    cutoff_p = float(sig.max())
    return float(-np.log10(max(cutoff_p, 1e-300)))


def _auto_label(ax, sub, *, top_k, always_label, label_format):
    """Stack labels at the left/right panel edges with leader lines.

    Selection rules:
      - Androgen axis: every significant (lab × stat) is labeled (no dedup).
      - Other categories: dedupe to the most-significant stat per lab,
        then include `always_label` whitelist + top_k by p-value.
    """
    sig = sub.loc[sub["sig"]].copy()
    if sig.empty:
        return

    androgen_rows = sig.loc[sig["category"] == "Androgen axis"]

    non_andro = sig.loc[sig["category"] != "Androgen axis"]
    best = (non_andro.sort_values("p_value")
                     .drop_duplicates("lab_name", keep="first"))
    always_sig = best.loc[best["lab_name"].isin(always_label)]
    extra = (best.loc[~best["lab_name"].isin(always_label)].head(top_k))
    non_andro_label = pd.concat([always_sig, extra]).drop_duplicates("lab_name")

    to_label = pd.concat([androgen_rows, non_andro_label]).drop_duplicates(
        subset=["lab_name", "feature_stat"])
    if to_label.empty:
        return

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    xspan = xlim[1] - xlim[0]
    yspan = ylim[1] - ylim[0]

    def _stack(side_df, side):
        if side_df.empty:
            return
        side_df = side_df.sort_values("neglog10p", ascending=False)
        if side == "left":
            label_x = xlim[0] + 0.04 * xspan
            ha = "left"
        else:
            label_x = xlim[1] - 0.04 * xspan
            ha = "right"
        last_y = None
        for _, r in side_df.iterrows():
            target_y = min(r["neglog10p"], Y_MAX_CAP)
            if last_y is not None and target_y > last_y - MIN_LABEL_SPACING_NLP:
                target_y = last_y - MIN_LABEL_SPACING_NLP
            if target_y < ylim[0] + 0.05 * yspan:
                continue
            last_y = target_y
            text = label_format.format(lab=r["lab_name"],
                                       stat=r["feature_stat"])
            color = CATEGORY_COLORS[r["category"]]
            ax.annotate(
                text, xy=(r["coef_feature"], min(r["neglog10p"], Y_MAX_CAP)),
                xytext=(label_x, target_y), textcoords="data",
                ha=ha, va="center",
                fontsize=LABEL_FONTSIZE, color=color, weight="semibold",
                arrowprops=dict(arrowstyle="-", lw=0.45, color="#95a5a6"),
                zorder=10,
            )

    left = to_label.loc[to_label["coef_feature"] < 0]
    right = to_label.loc[to_label["coef_feature"] >= 0]
    _stack(left, "left")
    _stack(right, "right")


def plot_volcano_panel(ax, sub, title):
    sub = sub.loc[~sub["lab_name"].isin(DROP)].copy()
    sub["category"] = sub["lab_name"].map(assign_category)
    sub["neglog10p"] = -np.log10(sub["p_value"].clip(lower=1e-300))
    sub["sig"] = sub["q_value"] < 0.05
    # Cap extreme -log10(p) so a handful of huge values don't squash the rest.
    sub["capped"] = sub["neglog10p"] > Y_MAX_CAP
    sub["y"] = sub["neglog10p"].clip(upper=Y_MAX_CAP)

    ns = sub.loc[~sub["sig"]]
    ax.scatter(ns["coef_feature"], ns["y"],
               s=20, color=NS_COLOR, alpha=0.45,
               edgecolor="none", zorder=1)

    for cat in DRAW_ORDER:
        cat_sig = sub.loc[sub["sig"] & (sub["category"] == cat)]
        if cat_sig.empty:
            continue
        is_hero = cat == "Androgen axis"
        base_kw = dict(
            color=CATEGORY_COLORS[cat],
            alpha=0.92,
            edgecolor="white", linewidth=0.6,
        )
        in_range = cat_sig.loc[~cat_sig["capped"]]
        if not in_range.empty:
            ax.scatter(
                in_range["coef_feature"], in_range["y"],
                s=80 if is_hero else 40, marker="o",
                zorder=5 if is_hero else 3, **base_kw,
            )
        over = cat_sig.loc[cat_sig["capped"]]
        if not over.empty:
            # Triangle marker only -- the exact p-value text callout was
            # dropped to keep this figure clean now that it stands alone.
            ax.scatter(
                over["coef_feature"], over["y"],
                s=110 if is_hero else 60, marker="^",
                zorder=6 if is_hero else 4, **base_kw,
            )

    ax.set_xlim(*PANEL_XLIM)
    y_max = float(sub["y"].max()) if not sub.empty else 5.0
    ax.set_ylim(-0.2, max(y_max * 1.10, 5.0))

    ax.axvline(0, color="grey", linestyle="-", linewidth=0.7, zorder=0)
    for x in (-0.5, 0.5):
        ax.axvline(x, color="grey", linestyle="--", linewidth=0.6,
                   alpha=0.7, zorder=0)
    q_y = q_threshold_neglog10p(sub)
    if q_y is not None:
        ax.axhline(q_y, color="black", linestyle=":", linewidth=0.9, zorder=0)

    _auto_label(ax, sub,
                top_k=TOP_K_PER_PANEL,
                always_label=ALWAYS_LABEL,
                label_format=LABEL_FORMAT)

    ax.set_xlabel("Cox log HR per SD")
    ax.set_ylabel(r"$-\log_{10}(p)$")
    ax.set_title(title, fontsize=12.5, weight="bold")

    n_tested = len(sub)
    n_sig = int(sub["sig"].sum())
    breakdown = (sub.loc[sub["sig"], "category"]
                 .value_counts()
                 .reindex([c for c in LEGEND_ORDER if c != "Other"], fill_value=0))
    short = {"Androgen axis": "Androgen", "CBC": "CBC", "LFT": "LFT",
             "CMP": "CMP", "Vitals": "Vitals"}
    bd_str = "  ".join(f"{short[c]} {int(n)}" for c, n in breakdown.items())
    ax.text(0.99, 0.02,
            f"{n_sig} / {n_tested} q<0.05   ·   {bd_str}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=8.5, color="#5d6d7e", family="monospace")


In [ ]:
# Load both landmarks and tag with landmark_days (authoritative from path).
def _load_uni(landmark):
    p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_univariate_nobs_adjusted.csv"
    df = pd.read_csv(p)
    df["landmark_days"] = landmark
    return df

uni = pd.concat([_load_uni(lm) for lm in LANDMARKS], ignore_index=True)
uni = uni.loc[uni["endpoint"] == "platinum"].copy()
uni = uni.dropna(subset=["coef_feature", "p_value", "q_value"])
print(f"{len(uni)} (lab × stat) rows across landmarks "
      f"{sorted(uni['landmark_days'].unique())}")

# Filter unstable Cox estimates: |log HR| > 4 or CI spans > 2 orders of magnitude.
COEF_CAP = 4.0
CI_RATIO_CAP = 100
ci_ratio = uni["ci_upper"] / uni["ci_lower"]
mask = (uni["coef_feature"].abs() <= COEF_CAP) & (ci_ratio < CI_RATIO_CAP)
print(f"dropping {int((~mask).sum())} / {len(uni)} unstable rows")
uni = uni.loc[mask].copy()
print(f"{len(uni)} rows remaining")

panels = [(0, "0 days"), (90, "+90 days")]
fig, axes = plt.subplots(
    1, 2, figsize=(14, 6),
    sharex=True, sharey=False,
    gridspec_kw={"wspace": 0.08},
)
for ax, (lm, title) in zip(axes, panels):
    sub = uni.loc[uni["landmark_days"] == lm]
    if sub.empty:
        ax.text(0.5, 0.5, f"(no data for landmark = {lm}d)",
                ha="center", va="center", transform=ax.transAxes,
                color="#7f8c8d")
        ax.set_axis_off()
        continue
    plot_volcano_panel(ax, sub, title)

handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
           for c in LEGEND_ORDER]
handles.append(Line2D([0], [0], marker="o", color="w",
                      markerfacecolor=NS_COLOR, markersize=8,
                      label="ns (q ≥ 0.05)"))
fig.legend(handles=handles, loc="lower center",
           ncol=len(handles), bbox_to_anchor=(0.5, -0.04),
           fontsize=10)

for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure3_univariate_platinum.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()


## Figure 4a — Discrimination vs. age baseline (AUC + C-index)

Two panels — **mean IPCW AUC(t)** and **C-index** on the held-out test fold — comparing each full labs model against its **age-only baseline** (hatched), on the **full cohort**, at both landmarks. Shows the lab-feature lift over an age-only predictor.

Sources (platinum endpoint):
- **Cox** `cox/landmark_{lm}/both/cox_agg_multivariable_metrics.csv` (labs) & `cox/landmark_{lm}/baseline/cox_agg_baseline_metrics.csv` (baseline) → `test_mean_auc_t`, `test_c_index`
- **XGBoost** `xgboost/landmark_{lm}/both/landmark_xgboost_metrics.csv` (labs) & `xgboost/landmark_{lm}/baseline/landmark_xgboost_baseline_metrics.csv` (baseline) → `mean_auc_t`, `c_index`

`render_discrimination_panel` (defined below) is reused as-is by the compiled Figure 4 later in this notebook.


In [ ]:
# Full labs models vs. their age-only baseline, on the full cohort.
def _read_metrics(path, auc_col, cindex_col):
    """Return (mean_auc_t, c_index) for the platinum endpoint, or (nan, nan)."""
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        return (np.nan, np.nan)
    row = df.loc[df["endpoint"] == "platinum"]
    if row.empty:
        return (np.nan, np.nan)
    row = row.iloc[0]
    return (float(row[auc_col]), float(row[cindex_col]))


def cox_labs(lm):
    return _read_metrics(BASE / "cox" / f"landmark_{lm}" / "both" / "cox_agg_multivariable_metrics.csv",
                         "test_mean_auc_t", "test_c_index")


def cox_baseline(lm):
    return _read_metrics(BASE / "cox" / f"landmark_{lm}" / "baseline" / "cox_agg_baseline_metrics.csv",
                         "test_mean_auc_t", "test_c_index")


def xgb_labs(lm):
    return _read_metrics(BASE / "xgboost" / f"landmark_{lm}" / "both" / "landmark_xgboost_metrics.csv",
                         "mean_auc_t", "c_index")


def xgb_baseline(lm):
    return _read_metrics(BASE / "xgboost" / f"landmark_{lm}" / "baseline" / "landmark_xgboost_baseline_metrics.csv",
                         "mean_auc_t", "c_index")


# (label, loader, color, hatch). Age baselines are the hatched, lighter twins.
DISCRIMINATION_SERIES = [
    ("Elastic-Net Cox",        cox_labs,     "#4C72B0", None),
    ("Cox baseline (age)",     cox_baseline, "#9DB3D6", "//"),
    ("XGBoost Survival",       xgb_labs,     "#B58900", None),
    ("XGBoost baseline (age)", xgb_baseline, "#E0CC8A", "//"),
]

# discrimination_data[label][k] = (auc, cindex) at LANDMARKS[k]
discrimination_data = {
    name: [loader(lm) for lm in LANDMARKS] for name, loader, _, _ in DISCRIMINATION_SERIES
}


def render_discrimination_panel(ax, *, ylabel, idx, data=discrimination_data,
                                series=DISCRIMINATION_SERIES, landmarks=LANDMARKS,
                                show_legend=False):
    """Draw one discrimination panel (AUC(t) or C-index) onto `ax`.

    Reused as-is by the standalone Figure 4a and by the compiled Figure 4.
    """
    n_series = len(series)
    bar_width = 0.19
    x_positions = np.arange(len(landmarks))
    offsets = (np.arange(n_series) - (n_series - 1) / 2) * bar_width

    for i, (name, _loader, color, hatch) in enumerate(series):
        vals = [data[name][k][idx] for k in range(len(landmarks))]
        bar_x = x_positions + offsets[i]
        ax.bar(bar_x, [v if np.isfinite(v) else 0.0 for v in vals], bar_width,
               color=color, hatch=hatch, edgecolor="white", linewidth=0.8, label=name)
        for x, v in zip(bar_x, vals):
            if np.isfinite(v):
                ax.text(x, v + 0.006, f"{v:.3f}", ha="center", va="bottom",
                        fontsize=7.5, weight="semibold", color=color)

    finite = [data[n][k][idx] for n in data for k in range(len(landmarks))
              if np.isfinite(data[n][k][idx])]
    ax.set_ylim(0.45, min(1.0, (max(finite) if finite else 0.7) * 1.12))
    ax.set_xticks(x_positions)
    ax.set_xticklabels([f"{('+' if lm > 0 else '')}{lm} days" for lm in landmarks])
    ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.9)
    ax.set_ylabel(ylabel)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, linestyle="--", alpha=0.3)
    if show_legend:
        ax.legend(loc="upper right", fontsize=8, ncol=1)


fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
render_discrimination_panel(axes[0], ylabel="Test Mean AUC(t)", idx=0, show_legend=True)
render_discrimination_panel(axes[1], ylabel="Test C-index", idx=1)
fig.suptitle("Labs vs. age baseline — platinum  ·  full cohort",
             fontsize=13, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])

for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure4a_discrimination_platinum.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()


## Figure 4b — 2×2 model importance grid

Top features by importance for Elastic-Net Cox (signed log HR coefficient, top row) and XGBoost Survival (gain, bottom row), one column per landmark. AGE is excluded so the focus is on lab-derived features.

Sources:
- **Cox**     `cox/landmark_{lm}/both/cox_agg_multivariable.csv`              → `endpoint == "platinum"`, column `coef`
- **XGBoost** `xgboost/landmark_{lm}/both/landmark_xgboost_feature_importance.csv` → `endpoint == "platinum"`, column `gain`

`render_importance_panel` (defined below) is reused as-is by the compiled Figure 4 later in this notebook.


In [ ]:
def load_cox_coefs(landmark):
    p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_multivariable.csv"
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "platinum"].copy()
    df = df.loc[~df["is_age_covariate"].fillna(False).astype(bool)]
    df = df.loc[df["coef"].fillna(0) != 0]
    return df


def load_xgb_importance(landmark):
    p = BASE / "xgboost" / f"landmark_{landmark}" / "both" / "landmark_xgboost_feature_importance.csv"
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "platinum"].copy()
    df = df.loc[df["feature"].str.lower() != "age"]
    df = df.loc[df["gain"].fillna(0) > 0]
    parsed = df["feature"].apply(lambda s: pd.Series(parse_feature(s),
                                                     index=["lab_name", "feature_stat"]))
    df = pd.concat([df.reset_index(drop=True), parsed.reset_index(drop=True)],
                   axis=1)
    return df


def render_importance_panel(ax, df, *, kind, title):
    if df.empty:
        ax.text(0.5, 0.5, "(no features to display)",
                ha="center", va="center", transform=ax.transAxes,
                color="#7f8c8d")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()
        return

    df = df.copy()
    df["category"] = df["lab_name"].map(assign_category)

    if kind == "cox":
        df = df.reindex(df["coef"].abs().sort_values(ascending=False).index).head(TOP_N)
        df = df.iloc[::-1]
        values = df["coef"].to_numpy()
        xlabel = "log HR coefficient"
    else:  # xgb
        df = df.sort_values("gain", ascending=False).head(TOP_N).iloc[::-1]
        values = df["gain"].to_numpy()
        xlabel = "XGBoost gain"

    colors = [CATEGORY_COLORS[c] for c in df["category"]]
    y = np.arange(len(df))
    ax.barh(y, values, color=colors, edgecolor="white", linewidth=0.6)

    if kind == "cox":
        ax.axvline(0, color="black", linewidth=0.5, zorder=0)

    labels = [format_label(r["lab_name"], r["feature_stat"])
              for _, r in df.iterrows()]
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8.5)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_title(title, fontsize=11, weight="bold")


IMPORTANCE_MODEL_ROWS = [
    ("cox", "Elastic-Net Cox", load_cox_coefs),
    ("xgb", "XGBoost Survival", load_xgb_importance),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9.5), constrained_layout=True)
for row, (kind, model_name, loader) in enumerate(IMPORTANCE_MODEL_ROWS):
    for col, lm in enumerate(LANDMARKS):
        ax = axes[row, col]
        try:
            df = loader(lm)
        except FileNotFoundError as e:
            ax.text(0.5, 0.5, f"file not found:\n{e.filename}",
                    ha="center", va="center", transform=ax.transAxes,
                    fontsize=8, color="#c0392b")
            ax.set_axis_off()
            continue
        sign = "+" if lm > 0 else ""
        title = f"{model_name}  ·  {sign}{lm} days"
        render_importance_panel(ax, df, kind=kind, title=title)

handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
           for c in LEGEND_ORDER]
fig.legend(handles=handles, loc="lower center",
           ncol=len(handles), bbox_to_anchor=(0.5, -0.04),
           fontsize=10)

for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure4b_importance_platinum.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()


## Figure 4 (compiled) — multivariate model performance

Assembles 4a (discrimination vs. age baseline) and 4b (2×2 importance grid) into one combined image for the manuscript, reusing `render_discrimination_panel` and `render_importance_panel` from the two sections above unchanged. Requires those two cells to have been run first (for `discrimination_data`, `DISCRIMINATION_SERIES`, `IMPORTANCE_MODEL_ROWS`, `load_cox_coefs`, `load_xgb_importance`).


In [ ]:
fig = plt.figure(figsize=(13, 13.5), constrained_layout=True)
gs = fig.add_gridspec(3, 2, height_ratios=[0.85, 1, 1])

# Row 0: discrimination (4a) -- two panels sharing the constrained-layout grid.
ax_auc = fig.add_subplot(gs[0, 0])
ax_cindex = fig.add_subplot(gs[0, 1])
render_discrimination_panel(ax_auc, ylabel="Test Mean AUC(t)", idx=0, show_legend=True)
render_discrimination_panel(ax_cindex, ylabel="Test C-index", idx=1)

# Rows 1-2: importance grid (4b) -- Cox row then XGBoost row, one column per landmark.
for row, (kind, model_name, loader) in enumerate(IMPORTANCE_MODEL_ROWS):
    for col, lm in enumerate(LANDMARKS):
        ax = fig.add_subplot(gs[row + 1, col])
        try:
            df = loader(lm)
        except FileNotFoundError as e:
            ax.text(0.5, 0.5, f"file not found:\n{e.filename}",
                    ha="center", va="center", transform=ax.transAxes,
                    fontsize=8, color="#c0392b")
            ax.set_axis_off()
            continue
        sign = "+" if lm > 0 else ""
        title = f"{model_name}  ·  {sign}{lm} days"
        render_importance_panel(ax, df, kind=kind, title=title)

handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
           for c in LEGEND_ORDER]
fig.legend(handles=handles, loc="lower center",
           ncol=len(handles), bbox_to_anchor=(0.5, -0.02),
           fontsize=10)
fig.suptitle("Figure 4 — Multivariate model performance (platinum)",
             fontsize=14, weight="bold")

for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure4_multivariate_platinum.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()


## Treatment-anchored arm

Uses the `t_treatment_anchor` outputs under `prediction_inputs_tx_anchor/` and `local_runs_tx_anchor/`.


## Paths

`BASE` points at the directory containing `cox/landmark_{lm}/...`, `xgboost/landmark_{lm}/...`, and `deephit/landmark_{lm}/...` outputs. `LONGITUDINAL_CSV` and `INPUTS_DIR` feed Figure 1 (cohort overview). Override all for local copies.


In [ ]:
NEPC_PROJ_PATH = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS")

BASE = NEPC_PROJ_PATH / "survival_analysis" / "local_runs_tx_anchor"
LONGITUDINAL_CSV = NEPC_PROJ_PATH / "longitudinal_prediction_data.csv"
INPUTS_DIR = NEPC_PROJ_PATH / "survival_analysis" / "prediction_inputs_tx_anchor"
AGGREGATED_CSV = INPUTS_DIR / "aggregated_landmark0.csv"  # optional stage enrichment for Table 1

OUT_DIR = Path("/data/gusev/USERS/jpconnor/figures/CAIA/COMPASS/tx_anchor")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LANDMARKS = [0, 90]
TOP_N = 15  # number of features shown per importance panel


## Figure 1 — Cohort overview

Three panels — each saved standalone, then assembled into one compiled image — plus a companion Table 1 (baseline characteristics, written separately as CSV + markdown, not a plotted panel):

- **Panel A — CONSORT attrition.** Reads the structured attrition counts `longitudinal_data_processing.py` writes to `cohort_attrition.json` (next to `LONGITUDINAL_CSV`) and `build_prediction_inputs.py` writes to `landmark_attrition.json` (in `INPUTS_DIR`) — no log-scraping.
- **Panel B — KM curve.** Platinum-free survival for the whole cohort (not landmark-rebased), from `LONGITUDINAL_CSV`.
- **Panel C — Timing histograms.** Record span, diagnosis → first treatment, and time to platinum (event patients only).

Table 1 covariates are limited to what PROFILE actually carries for the platinum analysis: Age, platinum exposure, follow-up time, plus optional cancer stage if `AGGREGATED_CSV` has `CANCER_STAGE_*` columns (only present if the pipeline run passed `--stage-file`). There is no race/ethnicity/insurance field in PROFILE.


In [ ]:
COHORT_LABEL = "PROFILE"
STAGE_COLUMNS = ["CANCER_STAGE_II", "CANCER_STAGE_III", "CANCER_STAGE_IV"]


def load_profile_patient_and_labs(path, *, id_col="DFCI_MRN"):
    """(patient_df, labs_df) split of longitudinal_prediction_data.csv."""
    df = pd.read_csv(path, low_memory=False)
    for c in ["DIAGNOSIS_DATE", "FIRST_TREATMENT_DATE", "PLATINUM_DATE",
              "LAST_CONTACT_DATE", "LAB_DATE", "FIRST_RECORD_DATE"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    if "t_dx_to_tx" not in df.columns and {"DIAGNOSIS_DATE", "FIRST_TREATMENT_DATE"} <= set(df.columns):
        df["t_dx_to_tx"] = (df["FIRST_TREATMENT_DATE"] - df["DIAGNOSIS_DATE"]).dt.days.astype("Float64")

    patient_level = [
        id_col, "AGE_AT_TREATMENTSTART", "FIRST_RECORD_DATE", "DIAGNOSIS_DATE",
        "FIRST_TREATMENT_DATE", "FIRST_TREATMENT", "LAST_CONTACT_DATE",
        "DEATH", "PLATINUM_MEDICATION", "PLATINUM_DATE", "PLATINUM",
        "t_diagnosis", "t_first_treatment", "t_treatment_anchor", "t_platinum",
        "t_last_contact", "t_death", "t_dx_to_tx",
    ]
    pat_cols = [c for c in patient_level if c in df.columns]
    patient_df = df[pat_cols].drop_duplicates(subset=[id_col]).reset_index(drop=True)

    lab_cols = [c for c in [id_col, "LAB_NAME", "LAB_VALUE", "LAB_UNIT", "LAB_DATE", "t_lab"]
                if c in df.columns]
    labs_df = df.loc[df["LAB_NAME"].notna(), lab_cols].reset_index(drop=True)
    return patient_df, labs_df


def record_span_days(labs_df, *, id_col, date_col="LAB_DATE"):
    g = labs_df.groupby(id_col)[date_col]
    span = (g.max() - g.min()).dt.days.astype("Float64")
    span.name = "record_span_days"
    return span


def load_attrition(longitudinal_csv, inputs_dir):
    cohort_path = longitudinal_csv.parent / "cohort_attrition.json"
    landmark_path = inputs_dir / "landmark_attrition.json"
    if not cohort_path.exists():
        raise FileNotFoundError(
            f"{cohort_path} not found -- re-run longitudinal_data_processing.py "
            "(it now writes cohort_attrition.json alongside its CSV outputs)."
        )
    if not landmark_path.exists():
        raise FileNotFoundError(
            f"{landmark_path} not found -- re-run build_prediction_inputs.py "
            "(it now writes landmark_attrition.json alongside its CSV outputs)."
        )
    return {**json.loads(cohort_path.read_text()), **json.loads(landmark_path.read_text())}


def render_consort_panel(ax, attrition):
    steps = [
        ("Patients with usable labs/vitals", attrition["n_with_labs"]),
        ("+ in death/last-contact table", attrition["n_after_death_table_join"]),
        ("+ recorded first treatment", attrition["n_after_first_treatment_filter"]),
        (f"+ ≥{attrition['min_psa_count']} PSA records", attrition["n_after_psa_count_filter"]),
        ("− PARPi-exposed excluded", attrition["n_after_parpi_exclusion"]),
    ]
    for landmark_str, n in sorted(attrition["eligible_by_landmark"].items(), key=lambda kv: int(kv[0])):
        sign = "+" if int(landmark_str) > 0 else ""
        steps.append((f"Eligible at landmark {sign}{landmark_str}d", n))
    steps.append(("Common cohort (all landmarks)", attrition["n_common_across_landmarks"]))

    n_steps = len(steps)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, n_steps + 0.4)
    ax.invert_yaxis()
    ax.axis("off")
    ax.set_title("Cohort attrition (CONSORT)", fontsize=12.5, weight="bold")

    box_w, box_h = 0.86, 0.62
    for i, (label, n) in enumerate(steps):
        y = i + (1 - box_h) / 2
        ax.add_patch(FancyBboxPatch(
            ((1 - box_w) / 2, y), box_w, box_h,
            boxstyle="round,pad=0.02,rounding_size=0.04",
            facecolor="#eef1f5", edgecolor="#5d6d7e", linewidth=1.0,
        ))
        ax.text(0.5, y + box_h / 2, f"{label}\nn = {n:,}",
                ha="center", va="center", fontsize=9)
        if i < n_steps - 1:
            ax.annotate(
                "", xy=(0.5, i + 1 + (1 - box_h) / 2), xytext=(0.5, y + box_h),
                arrowprops=dict(arrowstyle="-|>", color="#5d6d7e", lw=1.2),
            )

    split_sizes = attrition["split_sizes"]
    footer = "  ·  ".join(f"{k}: n={v:,}" for k, v in split_sizes.items())
    ax.text(0.5, n_steps + 0.15, f"Train/valid/test split — {footer}",
            ha="center", va="top", fontsize=8.5, color="#5d6d7e", style="italic")


def render_km_panel(ax, patient_df):
    from lifelines import KaplanMeierFitter

    dur = pd.to_numeric(patient_df["t_platinum"], errors="coerce")
    evt = pd.to_numeric(patient_df["PLATINUM"], errors="coerce")
    mask = dur.notna() & evt.notna() & (dur >= 0)
    kmf = KaplanMeierFitter(label=f"{COHORT_LABEL} (n={int(mask.sum()):,})")
    kmf.fit(durations=dur.loc[mask].to_numpy(dtype=float),
            event_observed=evt.loc[mask].to_numpy(dtype=float))
    kmf.plot_survival_function(ax=ax, color="#1f3a93", ci_show=True)
    ax.set_title("Platinum-free survival")
    ax.set_xlabel("Days from first record")
    ax.set_ylabel("Platinum-free probability")
    ax.set_ylim(0, 1.02)


def _overlay_hist(ax, series, *, bins=50):
    v = pd.to_numeric(series, errors="coerce").dropna().to_numpy(dtype=float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        ax.text(0.5, 0.5, "(no data)", ha="center", va="center", transform=ax.transAxes)
        return
    if isinstance(bins, int):
        lo, hi = np.percentile(v, [0.5, 99.5])
        if hi <= lo:
            lo, hi = v.min(), v.max() + 1.0
        bins = np.linspace(lo, hi, bins + 1)
    ax.hist(v, bins=bins, density=True, alpha=0.75, color="#1f3a93",
            label=f"{COHORT_LABEL} (n={v.size:,})", edgecolor="white", linewidth=0.3)
    ax.legend(fontsize=9, loc="best")


def render_timing_panel(axes, patient_df, labs_df, id_col):
    ax_span, ax_dx2tx, ax_tplat = axes

    span = record_span_days(labs_df, id_col=id_col)
    _overlay_hist(ax_span, span, bins=50)
    ax_span.set_xlabel("Record span (days)")
    ax_span.set_ylabel("Density")
    ax_span.set_title("Per-patient lab record span")

    _overlay_hist(ax_dx2tx, patient_df.get("t_dx_to_tx", pd.Series(dtype=float)), bins=50)
    ax_dx2tx.set_xlabel("Days: diagnosis → first treatment")
    ax_dx2tx.set_ylabel("Density")
    ax_dx2tx.set_title("Diagnosis → first treatment")

    is_event = pd.to_numeric(patient_df["PLATINUM"], errors="coerce").fillna(0).astype(int).astype(bool)
    _overlay_hist(ax_tplat, patient_df.loc[is_event, "t_platinum"], bins=40)
    ax_tplat.set_xlabel("Days to platinum, from first record (events only)")
    ax_tplat.set_ylabel("Density")
    ax_tplat.set_title("Time to platinum (event patients only)")


In [ ]:
def _mean_sd(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    return f"{s.mean():.1f} ± {s.std():.1f}" if not s.empty else "n/a"


def _median_iqr(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return "n/a"
    q1, med, q3 = s.quantile([0.25, 0.5, 0.75])
    return f"{med:.1f} ({q1:.1f}–{q3:.1f})"


def _count_pct(mask, total):
    n = int(mask.sum())
    return f"{n:,} ({100 * n / total:.1f}%)" if total else "n/a"


def _reconstruct_stage(aggregated_df):
    if not set(STAGE_COLUMNS).issubset(aggregated_df.columns):
        return None
    onehot = aggregated_df[STAGE_COLUMNS].apply(pd.to_numeric, errors="coerce")
    unknown = onehot.isna().all(axis=1)
    any_hit = (onehot.fillna(0) > 0).any(axis=1)
    picked = onehot.fillna(0).idxmax(axis=1).str[len("CANCER_STAGE_"):]
    stage = picked.where(any_hit, "I")
    return stage.mask(unknown, "Unknown")


def build_table1(patient_df, aggregated_df):
    n = len(patient_df)
    rows = [("N", f"{n:,}")]

    age = patient_df.get("AGE_AT_TREATMENTSTART", pd.Series(dtype=float))
    rows.append(("Age at first treatment, mean ± SD", _mean_sd(age)))
    rows.append(("Age at first treatment, median (IQR)", _median_iqr(age)))

    rows.append(("Platinum exposure, n (%)",
                 _count_pct(pd.to_numeric(patient_df["PLATINUM"], errors="coerce").fillna(0).astype(bool), n)))
    rows.append(("Median follow-up from first record, days (IQR)",
                 _median_iqr(patient_df.get("t_last_contact", pd.Series(dtype=float)))))

    if aggregated_df is None:
        rows.append(("Cancer stage", "not available (no AGGREGATED_CSV with CANCER_STAGE_* columns)"))
    else:
        n_agg = len(aggregated_df)
        stage = _reconstruct_stage(aggregated_df)
        if stage is not None:
            for cat in ["I", "II", "III", "IV", "Unknown"]:
                rows.append((f"  Cancer stage {cat}, n (%)", _count_pct(stage.eq(cat), n_agg)))
        else:
            rows.append(("Cancer stage", "not available in AGGREGATED_CSV"))

    return pd.DataFrame(rows, columns=["Characteristic", "Value"])


def _to_markdown_table(df):
    """Minimal GFM table renderer (no `tabulate` dependency, which isn't in
    this repo's pinned dependency list -- see README)."""
    header = "| " + " | ".join(df.columns) + " |"
    sep = "| " + " | ".join("---" for _ in df.columns) + " |"
    body = "\n".join("| " + " | ".join(str(v) for v in row) + " |"
                     for row in df.itertuples(index=False))
    return "\n".join([header, sep, body])


def write_table1(table1, out_base):
    out_base.parent.mkdir(parents=True, exist_ok=True)
    table1.to_csv(out_base.with_suffix(".csv"), index=False)
    out_base.with_suffix(".md").write_text(_to_markdown_table(table1))
    return [out_base.with_suffix(".csv"), out_base.with_suffix(".md")]


In [ ]:
ID_COL = "DFCI_MRN"

print(f"Loading attrition counts from {LONGITUDINAL_CSV.parent} and {INPUTS_DIR} ...")
attrition = load_attrition(LONGITUDINAL_CSV, INPUTS_DIR)

print(f"Loading {LONGITUDINAL_CSV} ...")
patient_df, labs_df = load_profile_patient_and_labs(LONGITUDINAL_CSV, id_col=ID_COL)
print(f"  patients={len(patient_df):,}  labs={len(labs_df):,}")

aggregated_df = None
if AGGREGATED_CSV.exists():
    aggregated_df = pd.read_csv(AGGREGATED_CSV, low_memory=False)
    print(f"Loaded {AGGREGATED_CSV} for Table 1 enrichment: {len(aggregated_df):,} rows")
else:
    print(f"[note] {AGGREGATED_CSV} not found; Table 1 will omit cancer stage")

# --- Panel A: CONSORT ---
n_landmarks = len(attrition["eligible_by_landmark"])
fig, ax = plt.subplots(figsize=(7.5, 0.62 * (n_landmarks + 8) + 1))
render_consort_panel(ax, attrition)
fig.tight_layout()
for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure1a_consort.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

# --- Panel B: KM curves ---
fig, ax = plt.subplots(figsize=(6.5, 4.8))
render_km_panel(ax, patient_df)
fig.tight_layout()
for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure1b_km.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

# --- Panel C: timing histograms ---
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
render_timing_panel(axes, patient_df, labs_df, ID_COL)
fig.tight_layout()
for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure1c_timing.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

# --- Compiled Figure 1 ---
fig = plt.figure(figsize=(14, 6 + 0.62 * (n_landmarks + 8)))
gs = fig.add_gridspec(3, 3, height_ratios=[0.62 * (n_landmarks + 8), 1.0, 1.0])
render_consort_panel(fig.add_subplot(gs[0, :]), attrition)
render_km_panel(fig.add_subplot(gs[1, :]), patient_df)
ax_span, ax_dx2tx, ax_tplat = (fig.add_subplot(gs[2, 0]), fig.add_subplot(gs[2, 1]), fig.add_subplot(gs[2, 2]))
render_timing_panel((ax_span, ax_dx2tx, ax_tplat), patient_df, labs_df, ID_COL)
fig.suptitle("Figure 1 — Cohort overview", fontsize=14, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.98])
for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure1_cohort_overview.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

# --- Table 1 ---
table1 = build_table1(patient_df, aggregated_df)
for p in write_table1(table1, OUT_DIR / "table1_baseline_characteristics"):
    print(f"wrote {p}")
table1


## Figure 3 — Paired volcano (univariate Cox)

Each dot is one (lab × feature_stat) pair. ns features (q ≥ 0.05) are faded gray; q-significant features are colored by their clinical category. Three narrative beats highlighted on the figure:

- **Testosterone** is a key indicator at both landmarks
- **PSA** becomes significant at +90d
- **CBC + LFT** signals (general health markers) emerge by +90d

Source: `cox/landmark_{lm}/both/cox_agg_univariate_nobs_adjusted.csv`.


In [ ]:
# ----------------------------- labeling knobs ---------------------------
TOP_K_PER_PANEL = 4
ALWAYS_LABEL = {"Hemoglobin", "Albumin", "Alkaline phosphatase"}
LABEL_FORMAT = "{lab} ({stat})"
LABEL_FONTSIZE = 8.5
MIN_LABEL_SPACING_NLP = 0.55
PANEL_XLIM = (-1.5, 1.5)
Y_MAX_CAP = 30  # -log10(p) ceiling; anything above this is drawn at the cap as a triangle


def q_threshold_neglog10p(sub):
    """y-value (-log10 p) at which q just crosses 0.05; None if no q-sig features."""
    sig = sub.loc[sub["q_value"] < 0.05, "p_value"]
    if sig.empty:
        return None
    cutoff_p = float(sig.max())
    return float(-np.log10(max(cutoff_p, 1e-300)))


def _auto_label(ax, sub, *, top_k, always_label, label_format):
    """Stack labels at the left/right panel edges with leader lines.

    Selection rules:
      - Androgen axis: every significant (lab × stat) is labeled (no dedup).
      - Other categories: dedupe to the most-significant stat per lab,
        then include `always_label` whitelist + top_k by p-value.
    """
    sig = sub.loc[sub["sig"]].copy()
    if sig.empty:
        return

    androgen_rows = sig.loc[sig["category"] == "Androgen axis"]

    non_andro = sig.loc[sig["category"] != "Androgen axis"]
    best = (non_andro.sort_values("p_value")
                     .drop_duplicates("lab_name", keep="first"))
    always_sig = best.loc[best["lab_name"].isin(always_label)]
    extra = (best.loc[~best["lab_name"].isin(always_label)].head(top_k))
    non_andro_label = pd.concat([always_sig, extra]).drop_duplicates("lab_name")

    to_label = pd.concat([androgen_rows, non_andro_label]).drop_duplicates(
        subset=["lab_name", "feature_stat"])
    if to_label.empty:
        return

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    xspan = xlim[1] - xlim[0]
    yspan = ylim[1] - ylim[0]

    def _stack(side_df, side):
        if side_df.empty:
            return
        side_df = side_df.sort_values("neglog10p", ascending=False)
        if side == "left":
            label_x = xlim[0] + 0.04 * xspan
            ha = "left"
        else:
            label_x = xlim[1] - 0.04 * xspan
            ha = "right"
        last_y = None
        for _, r in side_df.iterrows():
            target_y = min(r["neglog10p"], Y_MAX_CAP)
            if last_y is not None and target_y > last_y - MIN_LABEL_SPACING_NLP:
                target_y = last_y - MIN_LABEL_SPACING_NLP
            if target_y < ylim[0] + 0.05 * yspan:
                continue
            last_y = target_y
            text = label_format.format(lab=r["lab_name"],
                                       stat=r["feature_stat"])
            color = CATEGORY_COLORS[r["category"]]
            ax.annotate(
                text, xy=(r["coef_feature"], min(r["neglog10p"], Y_MAX_CAP)),
                xytext=(label_x, target_y), textcoords="data",
                ha=ha, va="center",
                fontsize=LABEL_FONTSIZE, color=color, weight="semibold",
                arrowprops=dict(arrowstyle="-", lw=0.45, color="#95a5a6"),
                zorder=10,
            )

    left = to_label.loc[to_label["coef_feature"] < 0]
    right = to_label.loc[to_label["coef_feature"] >= 0]
    _stack(left, "left")
    _stack(right, "right")


def plot_volcano_panel(ax, sub, title):
    sub = sub.loc[~sub["lab_name"].isin(DROP)].copy()
    sub["category"] = sub["lab_name"].map(assign_category)
    sub["neglog10p"] = -np.log10(sub["p_value"].clip(lower=1e-300))
    sub["sig"] = sub["q_value"] < 0.05
    # Cap extreme -log10(p) so a handful of huge values don't squash the rest.
    sub["capped"] = sub["neglog10p"] > Y_MAX_CAP
    sub["y"] = sub["neglog10p"].clip(upper=Y_MAX_CAP)

    ns = sub.loc[~sub["sig"]]
    ax.scatter(ns["coef_feature"], ns["y"],
               s=20, color=NS_COLOR, alpha=0.45,
               edgecolor="none", zorder=1)

    for cat in DRAW_ORDER:
        cat_sig = sub.loc[sub["sig"] & (sub["category"] == cat)]
        if cat_sig.empty:
            continue
        is_hero = cat == "Androgen axis"
        base_kw = dict(
            color=CATEGORY_COLORS[cat],
            alpha=0.92,
            edgecolor="white", linewidth=0.6,
        )
        in_range = cat_sig.loc[~cat_sig["capped"]]
        if not in_range.empty:
            ax.scatter(
                in_range["coef_feature"], in_range["y"],
                s=80 if is_hero else 40, marker="o",
                zorder=5 if is_hero else 3, **base_kw,
            )
        over = cat_sig.loc[cat_sig["capped"]]
        if not over.empty:
            # Triangle marker only -- the exact p-value text callout was
            # dropped to keep this figure clean now that it stands alone.
            ax.scatter(
                over["coef_feature"], over["y"],
                s=110 if is_hero else 60, marker="^",
                zorder=6 if is_hero else 4, **base_kw,
            )

    ax.set_xlim(*PANEL_XLIM)
    y_max = float(sub["y"].max()) if not sub.empty else 5.0
    ax.set_ylim(-0.2, max(y_max * 1.10, 5.0))

    ax.axvline(0, color="grey", linestyle="-", linewidth=0.7, zorder=0)
    for x in (-0.5, 0.5):
        ax.axvline(x, color="grey", linestyle="--", linewidth=0.6,
                   alpha=0.7, zorder=0)
    q_y = q_threshold_neglog10p(sub)
    if q_y is not None:
        ax.axhline(q_y, color="black", linestyle=":", linewidth=0.9, zorder=0)

    _auto_label(ax, sub,
                top_k=TOP_K_PER_PANEL,
                always_label=ALWAYS_LABEL,
                label_format=LABEL_FORMAT)

    ax.set_xlabel("Cox log HR per SD")
    ax.set_ylabel(r"$-\log_{10}(p)$")
    ax.set_title(title, fontsize=12.5, weight="bold")

    n_tested = len(sub)
    n_sig = int(sub["sig"].sum())
    breakdown = (sub.loc[sub["sig"], "category"]
                 .value_counts()
                 .reindex([c for c in LEGEND_ORDER if c != "Other"], fill_value=0))
    short = {"Androgen axis": "Androgen", "CBC": "CBC", "LFT": "LFT",
             "CMP": "CMP", "Vitals": "Vitals"}
    bd_str = "  ".join(f"{short[c]} {int(n)}" for c, n in breakdown.items())
    ax.text(0.99, 0.02,
            f"{n_sig} / {n_tested} q<0.05   ·   {bd_str}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=8.5, color="#5d6d7e", family="monospace")


In [ ]:
# Load both landmarks and tag with landmark_days (authoritative from path).
def _load_uni(landmark):
    p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_univariate_nobs_adjusted.csv"
    df = pd.read_csv(p)
    df["landmark_days"] = landmark
    return df

uni = pd.concat([_load_uni(lm) for lm in LANDMARKS], ignore_index=True)
uni = uni.loc[uni["endpoint"] == "platinum"].copy()
uni = uni.dropna(subset=["coef_feature", "p_value", "q_value"])
print(f"{len(uni)} (lab × stat) rows across landmarks "
      f"{sorted(uni['landmark_days'].unique())}")

# Filter unstable Cox estimates: |log HR| > 4 or CI spans > 2 orders of magnitude.
COEF_CAP = 4.0
CI_RATIO_CAP = 100
ci_ratio = uni["ci_upper"] / uni["ci_lower"]
mask = (uni["coef_feature"].abs() <= COEF_CAP) & (ci_ratio < CI_RATIO_CAP)
print(f"dropping {int((~mask).sum())} / {len(uni)} unstable rows")
uni = uni.loc[mask].copy()
print(f"{len(uni)} rows remaining")

panels = [(0, "0 days"), (90, "+90 days")]
fig, axes = plt.subplots(
    1, 2, figsize=(14, 6),
    sharex=True, sharey=False,
    gridspec_kw={"wspace": 0.08},
)
for ax, (lm, title) in zip(axes, panels):
    sub = uni.loc[uni["landmark_days"] == lm]
    if sub.empty:
        ax.text(0.5, 0.5, f"(no data for landmark = {lm}d)",
                ha="center", va="center", transform=ax.transAxes,
                color="#7f8c8d")
        ax.set_axis_off()
        continue
    plot_volcano_panel(ax, sub, title)

handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
           for c in LEGEND_ORDER]
handles.append(Line2D([0], [0], marker="o", color="w",
                      markerfacecolor=NS_COLOR, markersize=8,
                      label="ns (q ≥ 0.05)"))
fig.legend(handles=handles, loc="lower center",
           ncol=len(handles), bbox_to_anchor=(0.5, -0.04),
           fontsize=10)

for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure3_univariate_platinum.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()


## Figure 4a — Discrimination vs. age baseline (AUC + C-index)

Two panels — **mean IPCW AUC(t)** and **C-index** on the held-out test fold — comparing each full labs model against its **age-only baseline** (hatched), on the **treatment-anchored cohort**, at both landmarks. Shows the lab-feature lift over an age-only predictor.

Sources (platinum endpoint):
- **Cox** `cox/landmark_{lm}/both/cox_agg_multivariable_metrics.csv` (labs) & `cox/landmark_{lm}/baseline/cox_agg_baseline_metrics.csv` (baseline) → `test_mean_auc_t`, `test_c_index`
- **XGBoost** `xgboost/landmark_{lm}/both/landmark_xgboost_metrics.csv` (labs) & `xgboost/landmark_{lm}/baseline/landmark_xgboost_baseline_metrics.csv` (baseline) → `mean_auc_t`, `c_index`

`render_discrimination_panel` (defined below) is reused as-is by the compiled Figure 4 later in this notebook.


In [ ]:
# Full labs models vs. their age-only baseline, on the treatment-anchored cohort.
def _read_metrics(path, auc_col, cindex_col):
    """Return (mean_auc_t, c_index) for the platinum endpoint, or (nan, nan)."""
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        return (np.nan, np.nan)
    row = df.loc[df["endpoint"] == "platinum"]
    if row.empty:
        return (np.nan, np.nan)
    row = row.iloc[0]
    return (float(row[auc_col]), float(row[cindex_col]))


def cox_labs(lm):
    return _read_metrics(BASE / "cox" / f"landmark_{lm}" / "both" / "cox_agg_multivariable_metrics.csv",
                         "test_mean_auc_t", "test_c_index")


def cox_baseline(lm):
    return _read_metrics(BASE / "cox" / f"landmark_{lm}" / "baseline" / "cox_agg_baseline_metrics.csv",
                         "test_mean_auc_t", "test_c_index")


def xgb_labs(lm):
    return _read_metrics(BASE / "xgboost" / f"landmark_{lm}" / "both" / "landmark_xgboost_metrics.csv",
                         "mean_auc_t", "c_index")


def xgb_baseline(lm):
    return _read_metrics(BASE / "xgboost" / f"landmark_{lm}" / "baseline" / "landmark_xgboost_baseline_metrics.csv",
                         "mean_auc_t", "c_index")


# (label, loader, color, hatch). Age baselines are the hatched, lighter twins.
DISCRIMINATION_SERIES = [
    ("Elastic-Net Cox",        cox_labs,     "#4C72B0", None),
    ("Cox baseline (age)",     cox_baseline, "#9DB3D6", "//"),
    ("XGBoost Survival",       xgb_labs,     "#B58900", None),
    ("XGBoost baseline (age)", xgb_baseline, "#E0CC8A", "//"),
]

# discrimination_data[label][k] = (auc, cindex) at LANDMARKS[k]
discrimination_data = {
    name: [loader(lm) for lm in LANDMARKS] for name, loader, _, _ in DISCRIMINATION_SERIES
}


def render_discrimination_panel(ax, *, ylabel, idx, data=discrimination_data,
                                series=DISCRIMINATION_SERIES, landmarks=LANDMARKS,
                                show_legend=False):
    """Draw one discrimination panel (AUC(t) or C-index) onto `ax`.

    Reused as-is by the standalone Figure 4a and by the compiled Figure 4.
    """
    n_series = len(series)
    bar_width = 0.19
    x_positions = np.arange(len(landmarks))
    offsets = (np.arange(n_series) - (n_series - 1) / 2) * bar_width

    for i, (name, _loader, color, hatch) in enumerate(series):
        vals = [data[name][k][idx] for k in range(len(landmarks))]
        bar_x = x_positions + offsets[i]
        ax.bar(bar_x, [v if np.isfinite(v) else 0.0 for v in vals], bar_width,
               color=color, hatch=hatch, edgecolor="white", linewidth=0.8, label=name)
        for x, v in zip(bar_x, vals):
            if np.isfinite(v):
                ax.text(x, v + 0.006, f"{v:.3f}", ha="center", va="bottom",
                        fontsize=7.5, weight="semibold", color=color)

    finite = [data[n][k][idx] for n in data for k in range(len(landmarks))
              if np.isfinite(data[n][k][idx])]
    ax.set_ylim(0.45, min(1.0, (max(finite) if finite else 0.7) * 1.12))
    ax.set_xticks(x_positions)
    ax.set_xticklabels([f"{('+' if lm > 0 else '')}{lm} days" for lm in landmarks])
    ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.9)
    ax.set_ylabel(ylabel)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, linestyle="--", alpha=0.3)
    if show_legend:
        ax.legend(loc="upper right", fontsize=8, ncol=1)


fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
render_discrimination_panel(axes[0], ylabel="Test Mean AUC(t)", idx=0, show_legend=True)
render_discrimination_panel(axes[1], ylabel="Test C-index", idx=1)
fig.suptitle("Labs vs. age baseline — platinum  ·  treatment-anchored cohort",
             fontsize=13, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])

for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure4a_discrimination_platinum.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()


## Figure 4b — 2×2 model importance grid

Top features by importance for Elastic-Net Cox (signed log HR coefficient, top row) and XGBoost Survival (gain, bottom row), one column per landmark. AGE is excluded so the focus is on lab-derived features.

Sources:
- **Cox**     `cox/landmark_{lm}/both/cox_agg_multivariable.csv`              → `endpoint == "platinum"`, column `coef`
- **XGBoost** `xgboost/landmark_{lm}/both/landmark_xgboost_feature_importance.csv` → `endpoint == "platinum"`, column `gain`

`render_importance_panel` (defined below) is reused as-is by the compiled Figure 4 later in this notebook.


In [ ]:
def load_cox_coefs(landmark):
    p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_multivariable.csv"
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "platinum"].copy()
    df = df.loc[~df["is_age_covariate"].fillna(False).astype(bool)]
    df = df.loc[df["coef"].fillna(0) != 0]
    return df


def load_xgb_importance(landmark):
    p = BASE / "xgboost" / f"landmark_{landmark}" / "both" / "landmark_xgboost_feature_importance.csv"
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "platinum"].copy()
    df = df.loc[df["feature"].str.lower() != "age"]
    df = df.loc[df["gain"].fillna(0) > 0]
    parsed = df["feature"].apply(lambda s: pd.Series(parse_feature(s),
                                                     index=["lab_name", "feature_stat"]))
    df = pd.concat([df.reset_index(drop=True), parsed.reset_index(drop=True)],
                   axis=1)
    return df


def render_importance_panel(ax, df, *, kind, title):
    if df.empty:
        ax.text(0.5, 0.5, "(no features to display)",
                ha="center", va="center", transform=ax.transAxes,
                color="#7f8c8d")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()
        return

    df = df.copy()
    df["category"] = df["lab_name"].map(assign_category)

    if kind == "cox":
        df = df.reindex(df["coef"].abs().sort_values(ascending=False).index).head(TOP_N)
        df = df.iloc[::-1]
        values = df["coef"].to_numpy()
        xlabel = "log HR coefficient"
    else:  # xgb
        df = df.sort_values("gain", ascending=False).head(TOP_N).iloc[::-1]
        values = df["gain"].to_numpy()
        xlabel = "XGBoost gain"

    colors = [CATEGORY_COLORS[c] for c in df["category"]]
    y = np.arange(len(df))
    ax.barh(y, values, color=colors, edgecolor="white", linewidth=0.6)

    if kind == "cox":
        ax.axvline(0, color="black", linewidth=0.5, zorder=0)

    labels = [format_label(r["lab_name"], r["feature_stat"])
              for _, r in df.iterrows()]
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8.5)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_title(title, fontsize=11, weight="bold")


IMPORTANCE_MODEL_ROWS = [
    ("cox", "Elastic-Net Cox", load_cox_coefs),
    ("xgb", "XGBoost Survival", load_xgb_importance),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9.5), constrained_layout=True)
for row, (kind, model_name, loader) in enumerate(IMPORTANCE_MODEL_ROWS):
    for col, lm in enumerate(LANDMARKS):
        ax = axes[row, col]
        try:
            df = loader(lm)
        except FileNotFoundError as e:
            ax.text(0.5, 0.5, f"file not found:\n{e.filename}",
                    ha="center", va="center", transform=ax.transAxes,
                    fontsize=8, color="#c0392b")
            ax.set_axis_off()
            continue
        sign = "+" if lm > 0 else ""
        title = f"{model_name}  ·  {sign}{lm} days"
        render_importance_panel(ax, df, kind=kind, title=title)

handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
           for c in LEGEND_ORDER]
fig.legend(handles=handles, loc="lower center",
           ncol=len(handles), bbox_to_anchor=(0.5, -0.04),
           fontsize=10)

for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure4b_importance_platinum.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()


## Figure 4 (compiled) — multivariate model performance

Assembles 4a (discrimination vs. age baseline) and 4b (2×2 importance grid) into one combined image for the manuscript, reusing `render_discrimination_panel` and `render_importance_panel` from the two sections above unchanged. Requires those two cells to have been run first (for `discrimination_data`, `DISCRIMINATION_SERIES`, `IMPORTANCE_MODEL_ROWS`, `load_cox_coefs`, `load_xgb_importance`).


In [ ]:
fig = plt.figure(figsize=(13, 13.5), constrained_layout=True)
gs = fig.add_gridspec(3, 2, height_ratios=[0.85, 1, 1])

# Row 0: discrimination (4a) -- two panels sharing the constrained-layout grid.
ax_auc = fig.add_subplot(gs[0, 0])
ax_cindex = fig.add_subplot(gs[0, 1])
render_discrimination_panel(ax_auc, ylabel="Test Mean AUC(t)", idx=0, show_legend=True)
render_discrimination_panel(ax_cindex, ylabel="Test C-index", idx=1)

# Rows 1-2: importance grid (4b) -- Cox row then XGBoost row, one column per landmark.
for row, (kind, model_name, loader) in enumerate(IMPORTANCE_MODEL_ROWS):
    for col, lm in enumerate(LANDMARKS):
        ax = fig.add_subplot(gs[row + 1, col])
        try:
            df = loader(lm)
        except FileNotFoundError as e:
            ax.text(0.5, 0.5, f"file not found:\n{e.filename}",
                    ha="center", va="center", transform=ax.transAxes,
                    fontsize=8, color="#c0392b")
            ax.set_axis_off()
            continue
        sign = "+" if lm > 0 else ""
        title = f"{model_name}  ·  {sign}{lm} days"
        render_importance_panel(ax, df, kind=kind, title=title)

handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
           for c in LEGEND_ORDER]
fig.legend(handles=handles, loc="lower center",
           ncol=len(handles), bbox_to_anchor=(0.5, -0.02),
           fontsize=10)
fig.suptitle("Figure 4 — Multivariate model performance (platinum)",
             fontsize=14, weight="bold")

for ext in ("png", "pdf"):
    out = OUT_DIR / f"figure4_multivariate_platinum.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()
